In [1]:
import shutil
import pandas as pd
import random

import copy
import numpy as np # Import numpy for checking finite values
import matplotlib.pyplot as plt # Import matplotlib for potential debugging

import os
import math # Import math for ceil
from sklearn.manifold import TSNE # Import TSNE to check default perplexity



import os
import glob
import json
import numpy as np
import tensorflow as tf


In [3]:
import sys
utils_path = 'C:\\Users\\admin\\0.Master_Thesis\\'
sys.path.append(utils_path)

fixed_path = f'{utils_path}CellCNN'
#cell_repo = '/content/drive/MyDrive//Master Thesis/Code_trial_1/B-ALL_expression_matrix.csv'
if fixed_path not in sys.path:
    sys.path.append(fixed_path)

In [29]:
decache_files = ['cellCnn.model_new_unfixed', 'cellCnn.utils_new_unfixed', 'Dataset_Elaboration_Class', 'Dataset_utils', 'cellCnn.downsample_new_unfixed']
# Rimuovi il modulo specifico dalla cache

from Dataset_utils import remove_from_cache
remove_from_cache(decache_files)


# import downloaded modules
#from cellCnn.model import CellCnn
from cellCnn.model_new_unfixed import CellCnn
import cellCnn.plotting as plotting
import cellCnn.utils_new_unfixed as utils
#import cellCnn.utils as utils
import cellCnn.downsample_new_unfixed as downsample



from Dataset_Elaboration_Class import read_data, dataset_split, df_to_array, retrieve_labels
from Dataset_Elaboration_Class import patient_code_extraction, divide_donations,  idx_to_dataset
from Dataset_Elaboration_Class import class_division, train_val_samples, concatenate_hb_datasets, donor_division
from Dataset_Elaboration_Class import dataset_extraction, generate_subsets_length , subsample_generation, splitting

from Dataset_Elaboration_Class import run_CellCNN_res, dataset_elaboration, CellCNN_restructured, CV_CellCNN_restructured, CV_best_acc
from Dataset_utils import extract_hyper, phenotype_prediction, default_serializer, remove_from_cache, show_hyperparameters, min_length

from Dataset_utils import elaborate_predictions, show_hyper

cellCnn.model_new_unfixed rimosso dalla cache
cellCnn.utils_new_unfixed rimosso dalla cache
Dataset_Elaboration_Class rimosso dalla cache
Dataset_utils rimosso dalla cache
cellCnn.downsample_new_unfixed rimosso dalla cache


# Import datasets from .csv

In [5]:
%%time

# Trova tutti i file con estensione specifica in una cartella
data_folder = f'{utils_path}B-ALL_Datasets'
extension = '*.csv'  # cambia con l'estensione che ti serve

files_list = glob.glob(os.path.join(data_folder, extension))

ALL_DATASETS = []
counter = 0
multiple_donations = {}
no_id = []
for file_path in files_list:
    #import the dataset
    dataset = pd.read_csv(file_path, sep = ';', decimal = ',').astype('float32')
    ALL_DATASETS.append(dataset) # list of all datasets

    # divide the datasets by donors
    multiple_donations = patient_code_extraction(file_path, counter, multiple_donations)
    
    print(f"Elaborando file {counter}: {file_path}") # information about the process
    
    counter += 1 

# Fix no_id datasets
last_identifier = 0
for element in multiple_donations.keys():
    if element.isdigit():
        if int(element) > int(last_identifier):
            last_identifier = int(element)
print(last_identifier)
for dataset in multiple_donations['no_id']:
    last_identifier += 1
    multiple_donations[str(last_identifier)] = [dataset]
multiple_donations.pop('no_id')

Elaborando file 0: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220204-2900.csv
Elaborando file 1: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220208-3665.csv
Elaborando file 2: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220216-3546.csv
Elaborando file 3: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D15_1.csv
Elaborando file 4: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D29_1.csv
Elaborando file 5: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D71_1.csv
Elaborando file 6: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE12_D15_2.csv
Elaborando file 7: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE12_D29_1.csv
Elaborando file 8: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE1_D15_2.csv
Elaborando file 9: C

[0, 1, 2]

In [6]:
for i, dataset in enumerate(ALL_DATASETS):
    blast_n = (dataset['IsBlast'] == 1).sum()
    print(f'dataset {i} has {blast_n} cells over {len(dataset)} cells')

dataset 0 has 0 cells over 6558750 cells
dataset 1 has 0 cells over 2764877 cells
dataset 2 has 0 cells over 1291729 cells
dataset 3 has 1348 cells over 843500 cells
dataset 4 has 170 cells over 1404000 cells
dataset 5 has 0 cells over 3265250 cells
dataset 6 has 639 cells over 430869 cells
dataset 7 has 15 cells over 722000 cells
dataset 8 has 757 cells over 864000 cells
dataset 9 has 0 cells over 1947518 cells
dataset 10 has 308059 cells over 778750 cells
dataset 11 has 830101 cells over 5510750 cells
dataset 12 has 72 cells over 208000 cells
dataset 13 has 0 cells over 2912500 cells
dataset 14 has 3096 cells over 160500 cells
dataset 15 has 1449 cells over 3591480 cells
dataset 16 has 0 cells over 3107000 cells
dataset 17 has 9147 cells over 637409 cells
dataset 18 has 227 cells over 2928000 cells
dataset 19 has 518 cells over 164553 cells
dataset 20 has 398 cells over 191975 cells
dataset 21 has 0 cells over 169000 cells
dataset 22 has 2390 cells over 479000 cells
dataset 23 has 77

In [7]:
# Show patients donations
print(multiple_donations)
#['12', '9', '13', '7', '11'] 

{'11': [3, 4, 5], '12': [6, 7], '1': [8, 9], '2': [10, 11], '3': [12, 13], '4': [14, 15, 16], '6': [17, 18], '7': [19, 20, 21], '8': [22, 23], '9': [24, 25], '13': [0], '14': [1], '15': [2]}


In [8]:
healthy_donors, blast_donors, mixed_donors = donor_division(multiple_donations, ALL_DATASETS)
#print(first_subset)
#testing_set_idx = list(set(range(len(files_list))) - set(training_set_idx) - set(val_set_idx))
#print(training_set_idx, val_set_idx, testing_set_idx)
print(healthy_donors, blast_donors, mixed_donors)

{'11': [1, 1, 0], '12': [1, 1], '1': [1, 0], '2': [1, 1], '3': [1, 0], '4': [1, 1, 0], '6': [1, 1], '7': [1, 1, 0], '8': [1, 1], '9': [1, 1], '13': [0], '14': [0], '15': [0]}
['12', '2', '6', '8', '9'] ['13', '14', '15'] ['11', '1', '3', '4', '7']


In [9]:
def dataset_elaboration(multiple_donations, ALL_DATASETS, healthy_donors, blast_donors,
                        mixed_donors, n_sub = 3, seed = 42):
    """ Samples donors for Train, Validation and Test sets"""
    
    train_donors = []
    val_donors = []
    test_donors = []
    
    random.seed(seed)
    print(f'Precess starts. Dividing donors...')
    
    # sammple indexed for donor division
    healthy_donors_idx = random.sample(list(range(len(healthy_donors))), len(healthy_donors))
    blast_donors_idx = random.sample(list(range(len(blast_donors))), len(blast_donors))
    mixed_donors_idx = random.sample(list(range(len(mixed_donors))), len(mixed_donors))
    print(f'healthy_donors_idx, blast_donors_idx, mixed_donors_idx: {healthy_donors_idx}, {blast_donors_idx},{mixed_donors_idx}')

    print(f'Seting Train, Validation and Test idx...')
    # just divide accoding to the sampled indexes
    train_donors_idx, val_donors_idx, test_donors_idx = splitting(healthy_donors, blast_donors, mixed_donors, healthy_donors_idx, blast_donors_idx, mixed_donors_idx)
    print(train_donors_idx, val_donors_idx, test_donors_idx)

    return train_donors_idx, val_donors_idx, test_donors_idx

def donation_extraction(donors_idx, multiple_donations, ALL_DATASETS):
    """ Retrieves specific donors datasets (for ex. donors for train) from all datasets list """
    datasets_extracted = []
    for donor in donors_idx:
        donor_datasets = multiple_donations[donor]
        #print(donor_datasets)
        donor_donations = []
        for donation in donor_datasets:
            donation_dataset = ALL_DATASETS[donation].drop(columns = ['Original_ID'])
            
            donor_donations.append(donation_dataset)
            
        datasets_extracted.append(donor_donations)
    return datasets_extracted

def B_H_data_extraction(dataset, blast = True):
    """ Extracts blast or healthy cells from a specific dataset """
    code = 1 if blast == True else 0 
    data = pd.DataFrame()

    donation = dataset
    sub_data = donation[donation['IsBlast'] == code]
 
    data = pd.concat([data, sub_data], ignore_index = True)

    return data

def data_for_chunk(dataset, n_sub, blast = True):
    """ Compute the length of a chunk for Blast or Healthy datasets"""
    chunk_length = int(len(dataset) / n_sub)
              
    return chunk_length

def chunks_division(dataset, n_sub, blast = True, seed = 42):
    """ divides the dataset  in chunks creating a list of subsets of the dataset """
    if not blast:
        n_sub = n_sub*2

    
    chunk_length = data_for_chunk(dataset, n_sub = n_sub, blast = blast)
    
    #randomize data order
    dataset_shuffled = dataset.sample(frac=1, random_state=seed).reset_index(drop=True) # setting drop = False, it creates a new column with the original index of the ell

    data_chunks = [] 
    for i in range(n_sub):
        chunk_i = dataset_shuffled.iloc[i*chunk_length:  chunk_length * (i+1)]
        data_chunks.append(chunk_i)

    return data_chunks

def mixed_build_datasets(healthy_data_chunks, blast_data_chunks, n_cells = 100000, seed = 42):
    """ Builds the final datasets of a specific donor
        n_cells = number of healthy cells"""
    new_donor_datasets = []
    new_donor_y = []
    # blast_percentage_choice
    blast_percentages = [0.005, 0.01, 0.05, 0.1] #, 0.2] # 
    

    # dataset with blast cells
    for i in range(n_sub):
        # set randomicity
        seed = seed + 3*(i)
        random.seed(seed)
        n_healthy_cells = len(healthy_data_chunks[i]) # number of healthy cells
        
        blast_perc = random.choice(blast_percentages)
        n_blast_cells = int(blast_perc * n_cells) # number of blast cells
        
        # if number of cells is too high, the sample with replacement is activated
        if n_blast_cells > len(blast_data_chunks[i]):
            b_rep = True
        else:
            b_rep = False
    
        blast_data = blast_data_chunks[i].sample(n = n_blast_cells, replace = b_rep, random_state = seed).reset_index(drop=True)

        
        if n_cells > len(healthy_data_chunks[i]):
            h_rep = True
        else:
            h_rep = False
        healthy_data = healthy_data_chunks[i].sample(n = n_cells, replace = h_rep, random_state = seed).reset_index(drop=True)
        new_blast_dataset_i = pd.concat([healthy_data, blast_data], ignore_index = True)
        new_donor_datasets.append(new_blast_dataset_i)
        new_donor_y.append(1)
        print(f'New Artificial Blast Donation {i + 1}: length = {len(new_blast_dataset_i)}, healthy cells:{n_healthy_cells}, blast cells: {len(blast_data)}')

    # dataset with only healthy cells
    for i in range(n_sub, int(n_sub*2)):
        n_healthy_cells = len(healthy_data_chunks[i]) # number of healthy cells
        
        if n_cells > len(healthy_data_chunks[i]):
            h_rep = True
        else:
            h_rep = False
        # set randomicity
        seed = seed + 2*(i)
        random.seed(seed)
        
        
        healthy_data_i = healthy_data_chunks[i].sample(n = n_cells, replace = h_rep, random_state = seed).reset_index(drop=True)
        new_donor_datasets.append(healthy_data_i)
        new_donor_y.append(0)
        print(f'New Artificial Healthy Donation {i - n_sub + 1}: length = {len(healthy_data_i)}')

    return new_donor_datasets, new_donor_y
def healthy_build_datasets(healthy_data_chunks, n_cells = 100000, seed = 42):
    """ Builds the final datasets of a specific donor """
    new_donor_datasets = []
    new_donor_y = []
    # dataset with only healthy cells
    for i in range(n_sub, n_sub*2):
        # set randomicity
        seed = seed + 1*(i)
        random.seed(seed)
        
        n_healthy_cells = len(healthy_data_chunks[i]) # number of healthy cells
        
        if n_cells > len(healthy_data_chunks[i]):
            h_rep = True
        else:
            h_rep = False
        
        healthy_data_i = healthy_data_chunks[i].sample(n = n_cells, replace = h_rep, random_state = seed).reset_index(drop=True)
        new_donor_datasets.append(healthy_data_i)
        new_donor_y.append(0)
        print(f'New Artificial Healthy Donation {i - n_sub + 1}: length = {len(healthy_data_i)}')

    return new_donor_datasets, new_donor_y

def check_dataset_types(donor_datasets):
    counter = 0
    for dataset in donor_datasets:
        if len(dataset[dataset['IsBlast'] == 1]) > 0:
            counter += 1
    
    if counter > 0:
        type = 1
    else:
        type = 0
        
    #print(type)
    #print(f'\n')
    return type


def generate_new_datasets(donor_datasets_extracted, n_sub, n_cells, seed):
    """ generates new datasets from multiple donations of the same donor """
    
    blast_data = pd.DataFrame()
    healthy_data = pd.DataFrame()
   
    type = check_dataset_types(donor_datasets_extracted)
    #print(type)
    # aggregate healthy and blast data form all donor cells
    for dt in donor_datasets_extracted:
        healthy = len(dt[dt['IsBlast'] == 0])
        #print(healthy)
        if type == 1:
            blast_dataset_i = B_H_data_extraction(dt) #blast_data
            blast_data = pd.concat([blast_data, blast_dataset_i], ignore_index = True)

        
        healthy_dataset_i = B_H_data_extraction(dt, False)  #healthy_data
        #print(len(healthy_dataset_i))
        # create a single big dataset of blast or healthy cells
        healthy_data = pd.concat([healthy_data, healthy_dataset_i], ignore_index = True)
        #print(len(healthy_data))
    
    # shuffle anf divide the two datasets in chunks, from which extract final data
    if type == 1:
        blast_data_chunks = chunks_division(blast_data, n_sub = n_sub, blast = True, seed = seed)
        healthy_data_chunks = chunks_division(healthy_data, n_sub = n_sub, blast = False, seed = seed)
        
        #print(f'vhuk:{len(healthy_data_chunks[0])}')
    
        # create the new datasets
        new_donor_datasets, new_donor_y = mixed_build_datasets(healthy_data_chunks, blast_data_chunks, n_cells, seed = seed)
        
        
    else:
        healthy_data_chunks = chunks_division(healthy_data, n_sub = n_sub, blast = False, seed = seed)
        #print(len(healthy_data_chunks[0]))
        new_donor_datasets, new_donor_y = healthy_build_datasets(healthy_data_chunks, n_cells, seed = seed)

    return new_donor_datasets, new_donor_y



def splitting_and_dataset_elaboration(train_datasets_extracted, val_datasets_extracted, test_datasets_extracted, n_sub, n_cells, seed):
    new_train_datasets = []
    new_train_y = []
    
    new_val_datasets = []
    new_val_y = []
    
    new_test_datasets = []
    new_test_y = []

    print(f'New training datasets creation...')
    for donor_datasets in train_datasets_extracted:
        print(f'\nNew Donor')
        
        gen_results = generate_new_datasets(donor_datasets, n_sub, n_cells, seed)
        #for dt in gen_results
        new_train_datasets += gen_results[0]
        new_train_y += gen_results[1]
        seed += 1 
    print(new_train_y )
    print(f'Done!\n')
    
    
    print(f'New training datasets creation...')
    for donor_datasets in val_datasets_extracted:
        print(f'\nNew Donor')
        
        gen_results = generate_new_datasets(donor_datasets, n_sub, n_cells, seed)
        new_val_datasets += gen_results[0]
        new_val_y += gen_results[1]
        seed += 1 
    print(new_val_y )
    print(f'Done!\n')
    
    
    print(f'New training datasets creation...')
    for donor_datasets in test_datasets_extracted:
        print(f'\nNew Donor')


        gen_results = generate_new_datasets(donor_datasets, n_sub, n_cells, seed)
    
        new_test_datasets += gen_results[0]
        new_test_y += gen_results[1]
        seed += 1 
    print(new_test_y )
    print(f'Done!\n')

    return new_train_datasets, new_train_y, new_val_datasets, new_val_y, new_test_datasets, new_test_y

def remove_labels(new_test_datasets):
    no_labels = []
    for dataset in new_test_datasets:
        dataset = dataset.drop(columns = ['IsBlast'])
        no_labels.append(dataset)
    return no_labels

In [28]:

# Samples donors for Train, Validation and Test sets    
train_donors_idx, val_donors_idx, test_donors_idx = dataset_elaboration(multiple_donations, ALL_DATASETS, healthy_donors, blast_donors,
                        mixed_donors, seed = 42)



Precess starts. Dividing donors...
healthy_donors_idx, blast_donors_idx, mixed_donors_idx: [0, 4, 2, 1, 3], [0, 2, 1],[4, 0, 2, 1, 3]
Seting Train, Validation and Test idx...
['12', '9', '13', '7', '11'] ['6', '15', '3'] ['2', '8', '14', '1', '4']


In [21]:

#  Retrieves specific donor datasets from all datasets list
train_datasets_extracted = donation_extraction(train_donors_idx, multiple_donations, ALL_DATASETS)
val_datasets_extracted = donation_extraction(val_donors_idx, multiple_donations, ALL_DATASETS)
test_datasets_extracted = donation_extraction(test_donors_idx, multiple_donations, ALL_DATASETS)

#print(len(train_datasets_extracted)) # list of donators
#print(len(train_datasets_extracted[0])) # list of donations
print(len(test_datasets_extracted[0][0])) # dataset of cells

778750


In [14]:
print(len(test_datasets_extracted)) # dataset of cells

5


In [22]:
n_sub = 15
seed = 42
n_cells = 2000
new_train_datasets, new_train_y, new_val_datasets, new_val_y, new_test_datasets, new_test_y = splitting_and_dataset_elaboration(train_datasets_extracted, 
                                                                                    val_datasets_extracted, test_datasets_extracted,
                                                                                    n_sub, n_cells, seed)



New training datasets creation...

New Donor
New Artificial Blast Donation 1: length = 2010, healthy cells:38407, blast cells: 10
New Artificial Blast Donation 2: length = 2100, healthy cells:38407, blast cells: 100
New Artificial Blast Donation 3: length = 2020, healthy cells:38407, blast cells: 20
New Artificial Blast Donation 4: length = 2100, healthy cells:38407, blast cells: 100
New Artificial Blast Donation 5: length = 2010, healthy cells:38407, blast cells: 10
New Artificial Blast Donation 6: length = 2020, healthy cells:38407, blast cells: 20
New Artificial Blast Donation 7: length = 2100, healthy cells:38407, blast cells: 100
New Artificial Blast Donation 8: length = 2010, healthy cells:38407, blast cells: 10
New Artificial Blast Donation 9: length = 2100, healthy cells:38407, blast cells: 100
New Artificial Blast Donation 10: length = 2020, healthy cells:38407, blast cells: 20
New Artificial Blast Donation 11: length = 2020, healthy cells:38407, blast cells: 20
New Artificial

In [13]:
print(len(new_train_datasets))
print(len(new_train_y))
print(len(new_val_datasets))
print(len(new_val_y))
print(len(new_test_datasets))
print(len(new_test_y))


90
90
50
50
90
90


In [23]:
original_test_datasets = []
original_test_y = []
for donor in test_datasets_extracted:
    for dataset in donor:
        
        if len(dataset[dataset['IsBlast'] == 1]) > 0:
            original_test_y.append(1)
        else:
            original_test_y.append(0)
        dataset = dataset.drop(columns = ['IsBlast'])
        original_test_datasets.append(dataset)
print(len(original_test_datasets))
print(original_test_y)

10
[1, 1, 1, 1, 0, 1, 0, 1, 1, 0]


In [18]:
a = list(range(5))
b = list(range(5,10))
a += b
print(a)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


# Elaborate datasets

# CellCNN Restructured

In [19]:
new_test_datasets[0]

,FSC-A,FSC-H,SSC-A,CD81(FITC-A),CD73+CD304(PE-A),CD34(PerCP-Cy5-5-A),CD19(PE-Cy7-A),CD10(APC-A),CD38(APC-C750-A),CD20(PB-A),CD45(OC515-A),IsBlast
0,1.614500,1.911119,1.782181,0.773405,0.738709,1.174279,0.742864,0.745868,0.754686,1.140988,2.082722,0.0
1,1.136387,1.408263,0.482503,1.133935,2.881435,1.387460,2.573307,0.968724,0.200478,2.899436,2.719283,0.0
2,0.809383,1.112360,0.467287,0.683347,0.406240,1.299315,0.367376,0.407835,0.247154,0.434703,0.960288,0.0
3,0.658963,0.744562,0.863603,0.963120,0.495204,1.030112,0.262956,0.501908,0.301952,0.583014,0.696717,0.0
4,0.814603,0.958850,0.446276,1.250965,0.501360,1.429151,0.342837,0.772686,0.427225,0.663979,1.003243,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2195,0.688649,0.877053,0.523909,2.199861,1.832745,2.438559,0.747856,3.626066,0.943074,2.242818,2.053542,1.0
2196,1.167199,1.346009,0.762755,2.003672,1.403393,2.731042,1.763568,3.669773,1.757892,2.114295,2.092624,1.0
2197,0.701743,0.864924,0.471268,2.175352,3.372470,2.851587,1.488693,3.043298,0.166645,1.406805,1.848107,1.0
2198,0.549581,0.665746,0.453521,2.179678,1.427644,2.688058,0.985472,3.550565,-0.076017,2.100164,1.578255,1.0


In [24]:
new_no_label_test_datasets = []
for dataset in new_test_datasets:
    dataset = dataset.drop(columns = ['IsBlast'])
    new_no_label_test_datasets.append(dataset)
print(len(new_no_label_test_datasets))

135


In [33]:
%%time
seed = 42
seed_list = [7359, 9654, 4557, 106, 2615, 6924, 5574, 4552, 2547, 3527]
predictions_list, results_list = run_CellCNN_res(CellCnn, new_train_datasets, new_train_y,
                    new_val_datasets, new_val_y, 
                    new_no_label_test_datasets, 
                    trials = 10,
                    max_epochs = 1, nrun = 3, n_cell = 1000, seed_list = seed_list)

Trial 1 started
Seed used: 0
Model defined...
Fitting started...
=== BEFORE CATEGORICAL CONVERSION ===
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (135,)
n_classes: 1
=== AFTER CATEGORICAL CONVERSION ===
x_tr: (135, 1000, 11)
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (135, 2)
y_v type: <class 'numpy.ndarray'>
y_v shape: (75, 2)

# ============================================= #
Run: 0

Seed: 0
X_tr shape: (135, 1000, 11)
y_tr shape: (135, 2)
X_v shape: (75, 1000, 11)
y_v shape: (75, 2)
Unique values in y_tr: [0. 1.]
3/3 [==============================] - 0s 28ms/step - loss: 0.6867

# ============================================= #
Run: 1

Seed: 10
X_tr shape: (135, 1000, 11)
y_tr shape: (135, 2)
X_v shape: (75, 1000, 11)
y_v shape: (75, 2)
Unique values in y_tr: [0. 1.]
3/3 [==============================] - 0s 31ms/step - loss: 0.6778

# ============================================= #
Run: 2

Seed: 20
X_tr shape: (135, 1000, 11)
y_tr shape: (135, 2)
X_v shape: (75, 1000, 11)


C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib

5/5 [==============================] - 0s 17ms/step
Done
Trial 1 Done!

Trial 2 started
Seed used: 19308
Model defined...
Fitting started...
=== BEFORE CATEGORICAL CONVERSION ===
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (135,)
n_classes: 1
=== AFTER CATEGORICAL CONVERSION ===
x_tr: (135, 1000, 11)
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (135, 2)
y_v type: <class 'numpy.ndarray'>
y_v shape: (75, 2)

# ============================================= #
Run: 0

Seed: 19308
X_tr shape: (135, 1000, 11)
y_tr shape: (135, 2)
X_v shape: (75, 1000, 11)
y_v shape: (75, 2)
Unique values in y_tr: [0. 1.]
3/3 [==============================] - 0s 54ms/step - loss: 0.6898

# ============================================= #
Run: 1

Seed: 19318
X_tr shape: (135, 1000, 11)
y_tr shape: (135, 2)
X_v shape: (75, 1000, 11)
y_v shape: (75, 2)
Unique values in y_tr: [0. 1.]
3/3 [==============================] - 0s 36ms/step - loss: 0.6990

# ============================================= #
Run: 2

S

C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib

5/5 [==============================] - 0s 45ms/step
Done
Trial 2 Done!

Trial 3 started
Seed used: 18228
Model defined...
Fitting started...
=== BEFORE CATEGORICAL CONVERSION ===
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (135,)
n_classes: 1
=== AFTER CATEGORICAL CONVERSION ===
x_tr: (135, 1000, 11)
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (135, 2)
y_v type: <class 'numpy.ndarray'>
y_v shape: (75, 2)

# ============================================= #
Run: 0

Seed: 18228
X_tr shape: (135, 1000, 11)
y_tr shape: (135, 2)
X_v shape: (75, 1000, 11)
y_v shape: (75, 2)
Unique values in y_tr: [0. 1.]
3/3 [==============================] - 0s 25ms/step - loss: 0.7188

# ============================================= #
Run: 1

Seed: 18238
X_tr shape: (135, 1000, 11)
y_tr shape: (135, 2)
X_v shape: (75, 1000, 11)
y_v shape: (75, 2)
Unique values in y_tr: [0. 1.]
3/3 [==============================] - 0s 49ms/step - loss: 0.6722

# ============================================= #
Run: 2

S

...

5/5 [==============================] - 0s 18ms/step
Done
Trial 8 Done!

Trial 9 started
Seed used: 40752
Model defined...
Fitting started...
=== BEFORE CATEGORICAL CONVERSION ===
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (135,)
n_classes: 1
=== AFTER CATEGORICAL CONVERSION ===
x_tr: (135, 1000, 11)
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (135, 2)
y_v type: <class 'numpy.ndarray'>
y_v shape: (75, 2)

# ============================================= #
Run: 0

Seed: 40752
X_tr shape: (135, 1000, 11)
y_tr shape: (135, 2)
X_v shape: (75, 1000, 11)
y_v shape: (75, 2)
Unique values in y_tr: [0. 1.]
3/3 [==============================] - 0s 38ms/step - loss: 0.6174

# ============================================= #
Run: 1

Seed: 40762
X_tr shape: (135, 1000, 11)
y_tr shape: (135, 2)
X_v shape: (75, 1000, 11)
y_v shape: (75, 2)
Unique values in y_tr: [0. 1.]
3/3 [==============================] - 0s 51ms/step - loss: 0.6661

# ============================================= #
Run: 2

S

C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib

5/5 [==============================] - 0s 14ms/step
Done
Trial 9 Done!

Trial 10 started
Seed used: 63486
Model defined...
Fitting started...
=== BEFORE CATEGORICAL CONVERSION ===
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (135,)
n_classes: 1
=== AFTER CATEGORICAL CONVERSION ===
x_tr: (135, 1000, 11)
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (135, 2)
y_v type: <class 'numpy.ndarray'>
y_v shape: (75, 2)

# ============================================= #
Run: 0

Seed: 63486
X_tr shape: (135, 1000, 11)
y_tr shape: (135, 2)
X_v shape: (75, 1000, 11)
y_v shape: (75, 2)
Unique values in y_tr: [0. 1.]
3/3 [==============================] - 0s 58ms/step - loss: 0.6914

# ============================================= #
Run: 1

Seed: 63496
X_tr shape: (135, 1000, 11)
y_tr shape: (135, 2)
X_v shape: (75, 1000, 11)
y_v shape: (75, 2)
Unique values in y_tr: [0. 1.]
3/3 [==============================] - 0s 56ms/step - loss: 0.6793

# ============================================= #
Run: 2



C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib

5/5 [==============================] - 0s 16ms/step
Done
Trial 10 Done!

CPU times: total: 1min 15s
Wall time: 1min 37s


In [34]:
#best_per_trial = show_hyper(imported_new_results, best_3 = False)
#print(best_per_trial)
new_best_per_trial = show_hyper(results_list, best_3 = False)
print(new_best_per_trial)

   nfilter  learning_rate  maxpool_percentage
0      9.0          0.001                0.01
1      3.0          0.100                0.01
2      5.0          0.100                0.01
3      3.0          0.001              100.00
4      9.0          0.100                5.00
5      5.0          0.001                1.00
6      5.0          0.001                0.01
7      7.0          0.001               20.00
8      3.0          0.001                5.00
9      5.0          0.001                0.01


In [18]:
print(len(predictions_list[0]))


90


In [35]:
print(len(predictions_list[0]))
print(len(new_test_y))
pred_phenotype_df, accuracy_list = elaborate_predictions(predictions_list, new_test_y, results = True)

print(pred_phenotype_df.T)

135
135
Trial 0 Accuracy: 0.6444444444444445
Trial 1 Accuracy: 0.5555555555555556
Trial 2 Accuracy: 0.5555555555555556
Trial 3 Accuracy: 0.5555555555555556
Trial 4 Accuracy: 0.6148148148148148
Trial 5 Accuracy: 0.5555555555555556
Trial 6 Accuracy: 0.5555555555555556
Trial 7 Accuracy: 0.5555555555555556
Trial 8 Accuracy: 0.6962962962962963
Trial 9 Accuracy: 0.5555555555555556
             0    1    2    3    4    5    6    7    8    9    ...  125  126  \
0              0    0    0    0    0    0    0    0    0    0  ...    0    0   
1              0    0    0    0    0    0    0    0    0    0  ...    0    0   
2              0    0    0    0    0    0    0    0    0    0  ...    0    0   
3              0    0    0    0    0    0    0    0    0    0  ...    0    0   
4              1    1    1    1    1    1    1    1    1    1  ...    1    1   
5              0    0    0    0    0    0    0    0    0    0  ...    0    0   
6              0    0    0    0    0    0    0    0    0    0 

In [ ]:
'''
   nfilter  learning_rate  maxpool_percentage
0      7.0           0.01                0.01
1      7.0           0.01                0.01
2      5.0           0.01                0.01
3      9.0           0.01              100.00
4      5.0           0.01                1.00
5      7.0           0.01               20.00
6      7.0           0.01                0.01
7      3.0           0.01              100.00
8      5.0           0.01                0.01
9      5.0           0.01                0.01
===============================================
90
90
Trial 0 Accuracy: 0.6111111111111112
Trial 1 Accuracy: 0.5555555555555556
Trial 2 Accuracy: 0.5555555555555556
Trial 3 Accuracy: 0.6
Trial 4 Accuracy: 0.5555555555555556
Trial 5 Accuracy: 0.5555555555555556
Trial 6 Accuracy: 0.5555555555555556
Trial 7 Accuracy: 0.6777777777777778
Trial 8 Accuracy: 0.6666666666666666
Trial 9 Accuracy: 0.7666666666666667
             0   1   2   3   4   5   6   7   8   9   ...  80  81  82  83  84  \
0             1   0   0   0   0   0   0   1   0   0  ...   0   0   0   0   0   
1             0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   
2             0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   
3             1   0   0   0   0   0   0   1   0   0  ...   0   0   0   0   0   
4             0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   
5             0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   
6             0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   
7             1   0   0   1   0   0   0   1   0   0  ...   0   0   0   0   0   
8             1   0   0   1   0   0   0   1   0   0  ...   0   0   0   0   0   
9             1   0   0   1   1   0   0   1   0   0  ...   0   0   0   0   0   
True Labels   1   1   1   1   1   1   1   1   1   1  ...   0   0   0   0   0   

             85  86  87  88  89  
0             0   0   0   0   0  
1             0   0   0   0   0  
2             0   0   0   0   0  
3             0   0   0   0   0  
4             0   0   0   0   0  
5             0   0   0   0   0  
6             0   0   0   0   0  
7             0   0   0   0   0  
8             0   0   0   0   0  
9             0   0   0   0   0  
True Labels   0   0   0   0   0  

[11 rows x 90 columns]
mean_accuracy over the ten trials: 0.61
accuracy_std over the ten trials: 0.06875434890846659
    0  1  2  3  4  5  6  7  8  9  True Labels
0   1  0  0  1  0  0  0  1  1  1            1
1   0  0  0  0  0  0  0  0  0  0            1
2   0  0  0  0  0  0  0  0  0  0            1
3   0  0  0  0  0  0  0  1  1  1            1
4   0  0  0  0  0  0  0  0  0  1            1
.. .. .. .. .. .. .. .. .. .. ..          ...
85  0  0  0  0  0  0  0  0  0  0            0
86  0  0  0  0  0  0  0  0  0  0            0
87  0  0  0  0  0  0  0  0  0  0            0
88  0  0  0  0  0  0  0  0  0  0            0
89  0  0  0  0  0  0  0  0  0  0            0

[90 rows x 11 columns]
'''

In [88]:
%%time
seed_list = [7359, 9654, 4557, 106, 2615, 6924, 5574, 4552, 2547, 3527]
original_predictions_list, original_results_list = run_CellCNN_res(CellCnn, new_train_datasets, new_train_y,
                    new_val_datasets, new_val_y, 
                    original_test_datasets, #new_test_datasets, 
                    trials = 2,
                    max_epochs = 75, nrun = 10, n_cell = 1000, seed_list = seed_list)

Trial 1 started
Seed used: 0
Model defined...
Fitting started...
=== BEFORE CATEGORICAL CONVERSION ===
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (180,)
n_classes: 1
=== AFTER CATEGORICAL CONVERSION ===
x_tr: (180, 1000, 11)
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (180, 2)
y_v type: <class 'numpy.ndarray'>
y_v shape: (100, 2)

# ============================================= #
Run: 0

Seed: 0
X_tr shape: (180, 1000, 11)
y_tr shape: (180, 2)
X_v shape: (100, 1000, 11)
y_v shape: (100, 2)
Unique values in y_tr: [0. 1.]
Epoch 1/75
1/1 [==============================] - 0s 375ms/step - loss: 0.6933 - val_loss: 0.6895
Epoch 2/75
1/1 [==============================] - 0s 165ms/step - loss: 0.6895 - val_loss: 0.6863
Epoch 3/75
1/1 [==============================] - 0s 143ms/step - loss: 0.6878 - val_loss: 0.6839
Epoch 4/75
1/1 [==============================] - 0s 157ms/step - loss: 0.6943 - val_loss: 0.6827
Epoch 5/75
1/1 [==============================] - 0s 186ms/step - loss: 0.6

C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib

1/1 [==============================] - 0s 181ms/step
Done
Trial 1 Done!

Trial 2 started
Seed used: 19308000
Model defined...
Fitting started...
=== BEFORE CATEGORICAL CONVERSION ===
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (180,)
n_classes: 1
=== AFTER CATEGORICAL CONVERSION ===
x_tr: (180, 1000, 11)
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (180, 2)
y_v type: <class 'numpy.ndarray'>
y_v shape: (100, 2)

# ============================================= #
Run: 0

Seed: 19308000
X_tr shape: (180, 1000, 11)
y_tr shape: (180, 2)
X_v shape: (100, 1000, 11)
y_v shape: (100, 2)
Unique values in y_tr: [0. 1.]
Epoch 1/75
1/1 [==============================] - 0s 296ms/step - loss: 0.6921 - val_loss: 0.6852
Epoch 2/75
1/1 [==============================] - 0s 147ms/step - loss: 0.6875 - val_loss: 0.6802
Epoch 3/75
1/1 [==============================] - 0s 159ms/step - loss: 0.6845 - val_loss: 0.6749
Epoch 4/75
1/1 [==============================] - 0s 154ms/step - loss: 0.6822 - val_l

C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib

1/1 [==============================] - 0s 332ms/step
Done
Trial 2 Done!

CPU times: total: 5min 45s
Wall time: 4min 20s


In [89]:

tot_trials_res = []
par_list = ['config', 'model_sorted_idx']

for res in results_list:
    needed_results = {}
    for key, value in res.items():
        if key in par_list:
            print(key, value)
            needed_results[key] = value
    tot_trials_res.append(needed_results)
#print(tot_trials_res)

#print(predictions_list)

config {'nfilter': [3, 9, 7, 3, 5, 9, 7, 5, 3, 3], 'learning_rate': [], 'maxpool_percentage': [0.01, 1.0, 5.0, 20.0, 100.0, 0.01, 1.0, 5.0, 20.0, 100.0]}
model_sorted_idx [2 5 1 9 4 3 7 8 6 0]
config {'nfilter': [9, 7, 9, 5, 5, 3, 5, 7, 7, 3], 'learning_rate': [], 'maxpool_percentage': [0.01, 1.0, 5.0, 20.0, 100.0, 0.01, 1.0, 5.0, 20.0, 100.0]}
model_sorted_idx [5 8 3 4 9 7 6 0 2 1]


In [ ]:
'''

config {'nfilter': [3, 9, 7], 'learning_rate': [], 'maxpool_percentage': [0.01, 1.0, 5.0]}
model_sorted_idx [0 1 2]
config {'nfilter': [9, 7, 9], 'learning_rate': [], 'maxpool_percentage': [0.01, 1.0, 5.0]}
model_sorted_idx [0 1 2]
config {'nfilter': [5, 9, 7], 'learning_rate': [], 'maxpool_percentage': [0.01, 1.0, 5.0]}
model_sorted_idx [0 2 1]
'''

In [ ]:

import json
save_path = 'C:\\Users\\admin\\0.Master_Thesis\\CellCNN\\results\\'
#'''
with open(f'{save_path}new_results_list.json', "w", encoding="utf-8") as f:
    json.dump(tot_trials_res, f, default=default_serializer, ensure_ascii=False, indent=2)

with open(f'{save_path}new_predictions_list.json', "w", encoding="utf-8") as f:
    json.dump(predictions_list, f, default=default_serializer, ensure_ascii=False, indent=2)
#'''
with open(f'{save_path}new_predictions_list.json', "r", encoding="utf-8") as f:
    imported_new_predictions = json.load(f)

with open(f'{save_path}new_results_list.json', "r", encoding="utf-8") as f:
    imported_new_results = json.load(f)

In [ ]:
'''
print(results_list[0]['config'])

print(imported_new_results[0])

print(results_list[1]['config'])

print(imported_new_results[1])
'''

In [90]:
#best_per_trial = show_hyper(imported_new_results, best_3 = False)
#print(best_per_trial)
original_new_best_per_trial = show_hyper(original_results_list, best_3 = False)
print(original_new_best_per_trial)

   nfilter  learning_rate  maxpool_percentage
0      7.0            0.0                5.00
1      3.0            0.0                0.01


In [ ]:
   nfilter  learning_rate  maxpool_percentage
0      7.0            0.0                5.00
1      3.0            0.0                0.01

In [91]:
print(original_test_y)

original_pred_phenotype_df, original_accuracy_list = elaborate_predictions(original_predictions_list, original_test_y, results = True)

print(original_pred_phenotype_df.T)

[1, 1, 1, 1, 0, 1, 0, 1, 1, 0]
Trial 0 Accuracy: 0.3
Trial 1 Accuracy: 0.7
             0  1  2  3  4  5  6  7  8  9
0            0  0  0  0  0  0  0  0  0  0
1            1  1  0  0  0  1  0  1  0  0
True Labels  1  1  1  1  0  1  0  1  1  0
mean_accuracy over the ten trials: 0.5
accuracy_std over the ten trials: 0.19999999999999998
   0  1  True Labels
0  0  1            1
1  0  1            1
2  0  0            1
3  0  0            1
4  0  0            0
5  0  1            1
6  0  0            0
7  0  1            1
8  0  0            1
9  0  0            0


In [ ]:
[1, 1, 1, 1, 0, 1, 0, 1, 1, 0]
Trial 0 Accuracy: 0.3
Trial 1 Accuracy: 0.7
             0  1  2  3  4  5  6  7  8  9
0            0  0  0  0  0  0  0  0  0  0
1            1  1  0  0  0  1  0  1  0  0
True Labels  1  1  1  1  0  1  0  1  1  0
mean_accuracy over the ten trials: 0.5
accuracy_std over the ten trials: 0.19999999999999998
   0  1  True Labels
0  0  1            1
1  0  1            1
2  0  0            1
3  0  0            1
4  0  0            0
5  0  1            1
6  0  0            0
7  0  1            1
8  0  0            1
9  0  0            0

# Cross Validation Section

In [18]:
decache_files = ['cellCnn.model_new_unfixed', 'cellCnn.utils_new_unfixed', 'Dataset_Elaboration_Class', 'Dataset_utils', 'cellCnn.downsample_new_unfixed']
# Rimuovi il modulo specifico dalla cache

from Dataset_utils import remove_from_cache
remove_from_cache(decache_files)


# import downloaded modules
#from cellCnn.model import CellCnn
from cellCnn.model_new_unfixed import CellCnn
import cellCnn.plotting as plotting
import cellCnn.utils_new_unfixed as utils
#import cellCnn.utils as utils
import cellCnn.downsample_new_unfixed as downsample



from Dataset_Elaboration_Class import read_data, dataset_split, df_to_array, retrieve_labels
from Dataset_Elaboration_Class import patient_code_extraction, divide_donations,  idx_to_dataset
from Dataset_Elaboration_Class import class_division, train_val_samples, concatenate_hb_datasets, donor_division
from Dataset_Elaboration_Class import dataset_extraction, generate_subsets_length , subsample_generation, splitting

from Dataset_Elaboration_Class import run_CellCNN_res, dataset_elaboration, CellCNN_restructured, CV_CellCNN_restructured, CV_best_acc
from Dataset_utils import extract_hyper, phenotype_prediction, default_serializer, remove_from_cache, show_hyperparameters, min_length

from Dataset_utils import elaborate_predictions, show_hyper

cellCnn.model_new_unfixed rimosso dalla cache
cellCnn.utils_new_unfixed rimosso dalla cache
Dataset_Elaboration_Class rimosso dalla cache
Dataset_utils rimosso dalla cache
cellCnn.downsample_new_unfixed rimosso dalla cache


SyntaxError: invalid syntax (Dataset_Elaboration_Class.py, line 344)

In [ ]:
%%time
cv_new_train_datasets = new_train_datasets + new_val_datasets
cv_new_train_y = new_train_y + new_val_y
from sklearn.model_selection import StratifiedKFold
cv_results, model = CV_CellCNN_restructured(CellCnn, cv_train_datasets, cv_train_y, k = 3, n_cell = 10, n_sub = 3, seed = 42, 
                      max_epochs=1, patience=5, nrun=1)

In [ ]:

best_fold = CV_best_acc(cv_results) 
#print(best_fold)
# select the cofiguration of the fold that performed best
model.results = cv_results[best_fold]
print(model.results)
# predict using the selected model.results
cv_test_pred = [model.predict(test_datasets_no_labels)]

                    

In [ ]:
cv_tot_folds_res = []
par_list = ['config', 'model_sorted_idx']

for cv_res in cv_results:
    cv_needed_results = {}
    for key, value in cv_res.items():
        if key in par_list:
            print(key, value)
            cv_needed_results[key] = value
    cv_tot_folds_res.append(cv_needed_results)

#print(cv_tot_folds_res)

In [ ]:
import json
save_path = 'C:\\Users\\admin\\0.Master_Thesis\\CellCNN\\results\\'
#'''
with open(f'{save_path}cv_new_results_list.json', "w", encoding="utf-8") as f:
    json.dump(cv_tot_folds_res, f, default=default_serializer, ensure_ascii=False, indent=2)

with open(f'{save_path}cv_new_predictions_list.json', "w", encoding="utf-8") as f:
    json.dump(cv_test_pred, f, default=default_serializer, ensure_ascii=False, indent=2)
#'''
with open(f'{save_path}cv_new_predictions_list.json', "r", encoding="utf-8") as f:
    imported_cv_test_pred = json.load(f)

with open(f'{save_path}cv_new_results_list.json', "r", encoding="utf-8") as f:
    imported_cv_new_results = json.load(f)

In [ ]:
cv_best_per_fold = show_hyper(imported_cv_new_results, best_3 = False)
print(cv_best_per_fold)

In [ ]:
cv_pred_phenotype_df, cv_accuracy_list = elaborate_predictions( imported_cv_test_pred, test_y, results = True)